# Prune and Search

This techmique derives from Divide and Conquer. While in D&C we split into halves and recurse on boths, in Prune and Search we throw one of the halves away and recurse only on the other.

## BinarySearch

Given a sorted array, how do we find the index of an element?

1. **Divide**: Split the array in half
2. **Prune**: Compare the mid with the target and throw the half that can't possible contain the target
3. **Conquer**: Recurse on the other half

```
Algorithm BinarySearch(A, i, j, x)
Input: A sorted array A, starting index i ending index j, target x
Output: index of target x

if i == j:                // Base case: If there's only one element
    if A[i] = j:          // check if target
        return i          // and return.
    else:                 // O(1)
        return -1

mid = (j+1)/2             // Split in half. O(1)

if A[mid] == x:           // Prune and conquer.
    return mid            // Comparisons are O(1)
else if A[mid} > x:
    return BinarySearch(A, i, mid, x)   // Recursion is T(n/2)
else:
    return BinarySearch(A, mid + 1, j, x)
```

The time function is $T(n/2) + O(1)$

## Selection

Given an unsorted array, how do we find the k-th smallest element.

1. **Divide**: Split A into 2 arrays A1 and A2, around a pivot y
2. **Prune**: Check the size of A1, and throw away the subarray that doesn't have the k-th smallest
3. **Conquer**: Recurse on remaining subarray

```
Algorithm SelectionSeart(A, k)
Input: An array A, a target k
Output: The kth smallest element of A

if |A| <= 100:             // Base case: If we have 100 elements or less
    sort A                 // just sort and return
    return A[k]            // O(1) since the size is always 100 or smaller

y = some element of A      // Pick a pivot, O(?)

A1 = {}
A2 = {}

for each element e in A:           // Linear scan. O(n)
    if e < y:
        add e to Ai
    else if e > y:
        add e to A2

i = |A1|
if k = i + 1                   // If k is the element right after A1, is the pivot
    return y
else if k > i + 1                      // If k is an element after A1, recurse A2
    return SelectionSort(A2, k - i - 1) // But k is now a smaller element since we don't have A1
else:
    return SelectionSearch(A1, k)       // Prune and conquer: T(?)
```

How do we find the time function for this? Well, we first need to pick a pivot. We need the median of medians for that.

## Median of medians

Since in our selection algorithm the size of our subarrays depends on our pivot, the whole recurrence depends on how good the pivot is.

**Best case**: The pivot splits the subarrays in half, $T(n) = T(\frac{n}{2}) + n = n$

**Worst case**: The pivot is always the smallest or largest element, $T(n) = T(n-1) + O(n) = O(n^2)$

**Median of medians** is the clever answer to "how do we always pick a good pivot?". How can we guarantee that our pivot always splits the array into proportional subarrays? Obviously, the clear answer is to make the pivot a claue that's somewhere in the middle of the two subarrays, the **median** of A.

But, how do we find this median? A median of median algorithm is a workaround that finds an approximate median that is good enough to guarantee a balanced split.

- Uf we take a group of 5 elements, the median is "roughly in the middle" of those 5.
- If we take all the medians of various groups of 5 elements, the median of those medians is "roughly in the middle."

$3, 1, 4, 8, 2 | 9, 5, 7, 6, 10 | 11, 15, 12, 14, 13$

1. **Divide** into groups of 5 and find the median of each
    - Group 1: Sort -> $1, 2, 3, 4, 8$. Median = 3
    - Group 2: Sort -> $5, 6, 7, 9, 10$ Median = 7
    - Group 3: Sort -> $11, 12, 13, 14, 15$ Median = 13  
2. **Collect** the medians and find the median of them
   $1, 7, 13$ Median = 7
   
$1, 2, 3, 4, 5, 6 | 7 | 8, 9, 10, 11, 12, 13, 14, 15$

Dividing $n$ elements into $n/5$ groups, we get $n/5$ medians. The median of those medians (our pivot) is in the middle of those medians. Which means, at least half of the medians are smaller or equal to our pivot, so $n/10$ are smaller.

Also, each of those medians has at least 2 elements below it in the group the come from (since there are 5 elements), so that's $3n/10$ (since we also count the median itself). So, A1 has at least $3n/10$ elements, and A2 has $n - 3n/10 = 7n/10$ elements at most, and the opposite is also true.

Going back to our selection method, we get the recurrence $T(n) = T(\frac{7n}{10}) + T(\frac{n}{5}) + O(n)$, where $T(\frac{7n}{10})$ is recursing on the bigger possible part, $T(\frac{n}{5})$ is finding the median, and $O(n)$ is the partitioning scan.

```
Fiding the median:

B = {}                 // For our medians

for each group G of 5 elements in A:
    sort G                           // O(1) since group is always size 5
    add middle element of G to B     // Add the medians to B

y = Selection(B, |B|/2)          // Find the median of medians recursively, T(n/5)
```

## Weighted Selection

This algorithm is a generalization of the Selection problem. In this problem, every element has a weight, so instead of an array of elements, we have elements {x1...xn} and weights {w1...wn}, where $wi > 0$. Also instead of finding the k-th smallest element, we're given a target weight Zm and we want to find the element y such that:

- The total weight of anything **smaller** than y is $< z$.
- The total weight of anything **smaller than or equal** to y is $\geq Z$.

```
Elements: 1  3    2  6    9
Weights: 0.4 0.1 0.8 0.3 0.2
Z: 1.0
```

In a straightforward way, we can find the **weighted meadian** by sorting our elements, and looking at the middle element, and comparing the sum of the weights to Z before recursing, which would show us that the answe in 2. However, this method is $O(n \log n)$, and we can do better with Prune and Search, using the same method that in theoriginal Selection algorithm.

1. **Divide**: Pick a pivot y, and partition into A1 ($< y$), and A2 ($> y$).
2. **Prune**: Compute the total weight of A1. Based oon t and Z, decide which half cannot have the weighted median
3. **Conquer**: Recurse on the remaining half.

```
Algorithm WeightedSelection(A, Z)
Input: A map A with elements {x1...xn} and weights {w1...wn), a target weight Z
Output: An element y which is the weighted median

if |A| <= 100:             // Base case:
    sort A                 // O(1)
    loop through A to find the y where our cumulative weight >= Z
    return y            

y = Selection(A, |A|/2)      // Pick a pivot, O(n)

A1 = {}
A2 = {}

for each element e in A:           // Linear scan. O(n)
    if e.x < y.x:
        add e to A1
    else if e.x > y.x:
        add e to A2

sum = 0

for each element e in A1:       // O(7n/10)
    sum = sum + e.x=w
    
if sum < Z AND sum + y.w >= Z:       
    return y
else if sum < Z:
    return SelectionSort(A1, Z)
else if sum + y.w <= Z:
    return SelectionSearch(A2, Z - sum - y)
```

Just like in selection, our recurrence is $T(n) = T(\frac{5}{n}) + T(\frac{7n}{10}) + O(n)$.

## Multiple Selection